In [ ]:
# given a subj_id, iterate through its sessions and load the gs matrix
# select the best combo and save to plot across sessions and between regions
# if no difference across sessions, make median heatmap

In [ ]:
import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
from sg.eval_models import get_r2s_helper, get_num_units
import numpy as np

subj_ids = ["MR82", "MR83"]
sess_ids = ["20251027_152036", "20251027_162326"]

i = 0

get_num_units(subj_ids[i], sess_ids[i])
r2s = get_r2s_helper(subj_ids[i], sess_ids[i], folders=["all", "DMS", "DLS"], n_m=5)

In [ ]:
np.where(r2s["DLS"] == np.max(r2s["DLS"], axis=None))[0][0]

In [ ]:
from core.data import subject_ids, session_ids

subj_id = "MR82"
sess_ids = session_ids[np.where(subject_ids == subj_id)[0][0]]
folders = ["all", "ACC", "M2", "DMS", "DLS"]

best_combos = {key: [] for key in folders}
for sess_id in sess_ids:
    r2s = get_r2s_helper(subj_id, sess_id, folders=folders, n_m=5, do_plot=False)
    for key in r2s.keys():
        best_combo = np.where(r2s[key] == np.max(r2s[key], axis=None))
        best_combos[key].append((best_combo[0][0], best_combo[1][0]))

In [ ]:
# beta weight plots, 3x3 scatterplot errorbars
# rerun without cid enforcement and show bar plots
# firing rate scatter
# grid search rerun

In [ ]:
# for a given subject
# regions are cols, epochs are rows
# get the per neuron beta weight across the sessions and seeds for mb and mf
# mean and std, scatter

In [ ]:
from utils.paths import MODELS_DIR, FIGURES_DIR
import pickle

subj_id = "MR82"
sess_ids = session_ids[np.where(subject_ids == subj_id)[0][0]]

beta_weights = {}

n_cvs = 5
regions = ["all", "DMS", "DLS"]
epoch_keys = ["choice", "reward", "iti"]

tv = "response"
betas_sess = []

seed = 0
for sess_id in sess_ids:
    no_families = False
    betas = {
        reg: {
            epoch: {strategy: [] for strategy in ["mb", "mf"]} for epoch in epoch_keys
        }
        for reg in regions
    }
    # load the families
    file_path = (
        MODELS_DIR
        / "fit"
        / subj_id
        / sess_id
        / "river_n_tributaries"
        / "results_dict.pkl"
    )

    if not file_path.is_file():
        continue

    with open(file_path, "rb") as f:
        res_dict = pickle.load(f)
    for reg in regions:
        print(">", reg)
        for epoch in epoch_keys:
            print(">>", epoch)
            families_mb = res_dict[reg][epoch]["mb"]["families"]
            families_mf = res_dict[reg][epoch]["mf"]["families"]

            if len(families_mb) == 0:
                no_families = True
                break
            beta_mb_sess = []
            beta_mf_sess = []

            family_mb = families_mb[seed]
            family_mf = families_mf[seed]

            beta_mb = family_mb.mod_taskvar.tv.weight.data[:]
            beta_mf = family_mf.mod_taskvar.tv.weight.data[:]

            tv_idxs = []
            tv_labels = []
            counter = 0
            for tv_ in family_mb.task_vars:
                for val in family_mb.trial_data[tv_].unique():
                    if tv_ == tv:
                        tv_idxs.append(counter)
                        if tv_ == "response":
                            if val == 1:
                                val_str = "left"
                            elif val == -1:
                                val_str = "right"
                        if tv_ == "rewarded":
                            if val == 1:
                                val_str = "correct"
                            elif val == 0:
                                val_str = "incorrect"
                        tv_labels.append(f"{tv_}_{val_str}")
                    counter += 1

            # beta_mb_sess.append([beta_mb[tv_idx] for tv_idx in tv_idxs])
            # beta_mf_sess.append([beta_mf[tv_idx] for tv_idx in tv_idxs])
            betas[reg][epoch]["mb"] = np.array([beta_mb[tv_idx] for tv_idx in tv_idxs])
            betas[reg][epoch]["mf"] = np.array([beta_mf[tv_idx] for tv_idx in tv_idxs])
    if not no_families:
        betas_sess.append(betas)

In [ ]:
num_tvs = np.shape(betas_sess[0]["all"]["choice"]["mb"])[0]

colors_tv = ["#17855D", "#9554BD"]

fig, axes = plt.subplots(ncols=3, nrows=3, figsize=(5, 5))

for i, reg in enumerate(regions):
    for j, epoch in enumerate(epoch_keys):
        ax = axes[j][i]

        for betas in betas_sess:
            betas_mb = betas[reg][epoch]["mb"]
            betas_mf = betas[reg][epoch]["mf"]

            for tv_idx in range(num_tvs):
                ax.scatter(
                    betas_mb[tv_idx],
                    betas_mf[tv_idx],
                    s=0.5,
                    color=colors_tv[tv_idx],
                    alpha=0.5,
                    label=f"{tv_labels[tv_idx]}",
                )

        ax.axhline(y=0, linewidth=0.5, color="#888888", linestyle="--")
        ax.axvline(x=0, linewidth=0.5, color="#888888", linestyle="--")
        ax.set_xlim([-2, 2])
        ax.set_ylim([-2, 2])
        ax.set_xlabel(r"$\beta$ mb")
        ax.set_ylabel(r"$\beta$ mf")

fig.suptitle(tv)
fig.tight_layout()

fpath_png = FIGURES_DIR / "beta_strategy" / subj_id / f"{tv}_beta_strategy-{seed}.png"
fpath_svg = FIGURES_DIR / "beta_strategy" / subj_id / f"{tv}_beta_strategy-{seed}.svg"
fpath_png.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(fpath_png, dpi=300, bbox_inches="tight")
fig.savefig(fpath_svg, dpi=300, bbox_inches="tight")

In [ ]:
from core.data import subject_ids, session_ids
from utils.paths import PROJECT_ROOT
import pickle
import numpy as np

subj_id = "MM012"

corr_dicts = []

for sess_id in session_ids[np.where(subject_ids == subj_id)[0][0]]:
    save_path = PROJECT_ROOT.parents[0] / "corr" / subj_id / sess_id / "sncorr.pkl"

    with open(save_path, "rb") as f:
        corr_dict = pickle.load(f)

    corr_dicts.append(corr_dict)

In [ ]:
from scipy.stats import sem

regions = ["ACC", "M2", "DMS", "DLS"]
region_pairs = [f"{reg1}-{reg2}" for reg1 in regions for reg2 in regions]

scorr_mean = {
    reg_pair: np.mean(
        [
            np.mean(corr_dict["scorr"][reg_pair])
            for corr_dict in corr_dicts
            if reg_pair in corr_dict["scorr"].keys()
        ]
    )
    for reg_pair in region_pairs
}
scorr_med = {
    reg_pair: np.mean(
        [
            np.median(corr_dict["scorr"][reg_pair])
            for corr_dict in corr_dicts
            if reg_pair in corr_dict["scorr"].keys()
        ]
    )
    for reg_pair in region_pairs
}
scorr_sem = {
    reg_pair: np.mean(
        [
            sem(corr_dict["scorr"][reg_pair], axis=None)
            for corr_dict in corr_dicts
            if reg_pair in corr_dict["scorr"].keys()
        ]
    )
    for reg_pair in region_pairs
}

ncorr_mean = {
    reg_pair: np.mean(
        [
            np.mean(corr_dict["ncorr"][reg_pair])
            for corr_dict in corr_dicts
            if reg_pair in corr_dict["ncorr"].keys()
        ]
    )
    for reg_pair in region_pairs
}
ncorr_med = {
    reg_pair: np.mean(
        [
            np.median(corr_dict["ncorr"][reg_pair])
            for corr_dict in corr_dicts
            if reg_pair in corr_dict["ncorr"].keys()
        ]
    )
    for reg_pair in region_pairs
}
ncorr_sem = {
    reg_pair: np.mean(
        [
            sem(corr_dict["ncorr"][reg_pair], axis=None)
            for corr_dict in corr_dicts
            if reg_pair in corr_dict["ncorr"].keys()
        ]
    )
    for reg_pair in region_pairs
}

In [ ]:
scorr_med

In [ ]:
from utils.paths import FIGURES_DIR
import matplotlib.pyplot as plt


def plot_sncorr_bar(corr_m, corr_s, corr_label, regions):
    x = np.arange(len(regions))

    fig, ax = plt.subplots(figsize=(5, 5))
    width = 0.2

    if "scorr" in corr_label:
        color = "#27570F"
    elif "ncorr" in corr_label:
        color = "#9C3232"
    else:
        color = "#666666"

    tick_locs = []
    tick_labels = []

    for i, reg_a in enumerate(regions):
        corr_m_ = [corr_m[f"{reg_a}-{reg_b}"] for reg_b in regions]
        corr_s_ = [corr_s[f"{reg_a}-{reg_b}"] for reg_b in regions]

        ax.bar(
            x + (width * i),
            corr_m_,
            width=0.18,
            linewidth=1,
            edgecolor="k",
            color=color,
        )
        ax.errorbar(x + (width * i), corr_m_, corr_s_, capsize=2, fmt=".", color="k")

        tick_locs.extend(x + (width * i))
        tick_labels.extend([f"{reg_a}-{reg_b}" for reg_b in regions])

    ax.set_xticks(
        np.sort(tick_locs), np.array(tick_labels)[np.argsort(tick_locs)], rotation=90
    )
    ax.set_ylabel(corr_label)

    fpath_png = FIGURES_DIR / "sncorr" / subj_id / f"{corr_label}.png"
    fpath_svg = FIGURES_DIR / "sncorr" / subj_id / f"{corr_label}.svg"
    fpath_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(fpath_png, dpi=300, bbox_inches="tight")
    fig.savefig(fpath_svg, dpi=300, bbox_inches="tight")

In [ ]:
plot_sncorr_bar(
    ncorr_mean, ncorr_sem, corr_label="ncorr_mean", regions=["ACC", "M2", "DMS", "DLS"]
)